In [1]:
import pandas as pd

# Membaca file yang diupload langsung
df = pd.read_csv('cdc_diabetes_health_indicators.csv')

# Intip data untuk memastikan sudah terbaca
print(df.head())

   Diabetes_binary  HighBP  HighChol  CholCheck   BMI  Smoker  Stroke  \
0              0.0     1.0       0.0        1.0  26.0     0.0     0.0   
1              0.0     1.0       1.0        1.0  26.0     1.0     1.0   
2              0.0     0.0       0.0        1.0  26.0     0.0     0.0   
3              0.0     1.0       1.0        1.0  28.0     1.0     0.0   
4              0.0     0.0       0.0        1.0  29.0     1.0     0.0   

   HeartDiseaseorAttack  PhysActivity  Fruits  ...  AnyHealthcare  \
0                   0.0           1.0     0.0  ...            1.0   
1                   0.0           0.0     1.0  ...            1.0   
2                   0.0           1.0     1.0  ...            1.0   
3                   0.0           1.0     1.0  ...            1.0   
4                   0.0           1.0     1.0  ...            1.0   

   NoDocbcCost  GenHlth  MentHlth  PhysHlth  DiffWalk  Sex   Age  Education  \
0          0.0      3.0       5.0      30.0       0.0  1.0   4.0   

# Running Data

In [ ]:
import pandas as pd
import numpy as np
import joblib
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score

# 1. Pra-pemrosesan Data
df = df.dropna()  # Hapus baris jika ada nilai kosong

# Memisahkan Fitur (X) dan Target (y)
# (Sesuaikan 'Diabetes_binary' dengan nama kolom target asli di datasetmu)
X = df.drop(columns=['Diabetes_binary'])
y = df['Diabetes_binary']

# 2. Splitting Data (70% Latih, 30% Uji)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.30, random_state=42)

# 3. Inisialisasi Model
#    RF dibatasi (max_depth / min_samples_leaf) supaya ukuran file pickle wajar
#    (RF default bisa ratusan MB) tanpa menurunkan akurasi secara berarti.
rf_model = RandomForestClassifier(
    n_estimators=100, max_depth=12, min_samples_leaf=25, random_state=42, n_jobs=-1)
lr_model = LogisticRegression(max_iter=1000, random_state=42)
gb_model = GradientBoostingClassifier(random_state=42)

# 4. Running Training (Proses Pelatihan)
print("Sedang melatih model... Mohon tunggu karena dataset CDC cukup besar.")
rf_model.fit(X_train, y_train)
lr_model.fit(X_train, y_train)
gb_model.fit(X_train, y_train)
print("Pelatihan selesai!\n")

# 5. Evaluasi & Cetak Hasil Perbandingan
models = {
    "Random Forest": rf_model,
    "Logistic Regression": lr_model,
    "Gradient Boosting": gb_model
}

best_name, best_acc = None, -1.0
for nama_model, model in models.items():
    y_pred = model.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    print(f"=== {nama_model} ===")
    print(f"Akurasi: {acc * 100:.2f}%")
    print(classification_report(y_test, y_pred))
    print("-" * 40)
    if acc > best_acc:
        best_name, best_acc = nama_model, acc

# 6. Simpan KETIGA model agar aplikasi web bisa menampilkan skor nyata tiap model
joblib.dump(rf_model, 'model_rf.pkl')
joblib.dump(lr_model, 'model_lr.pkl')
joblib.dump(gb_model, 'model_gb.pkl')

# 7. "3 jadi 1": pilih model terbaik (akurasi tertinggi) sebagai model utama aplikasi.
#    Di sini Gradient Boosting yang menang, jadi disalin sebagai model_diabetes_terbaik.pkl.
best_model = models[best_name]
joblib.dump(best_model, 'model_diabetes_terbaik.pkl')
print(f"Model terbaik: {best_name} ({best_acc * 100:.2f}%) disimpan sebagai 'model_diabetes_terbaik.pkl'")
print("Ketiga model (rf/lr/gb) juga disimpan untuk perbandingan di web app.")
